# Materials 06 — Nanoindentation

The final experiment of the series: press a rigid spherical **indenter**
into a free surface and record the load–displacement curve — the atomistic
version of the workhorse experiment of small-scale mechanics, and the
original Tutorial 6 minus the EAM potential and STL nano-stamping.

Surface simulations need thermostat care (the original tutorial's pattern,
reproduced here): the slab is split into a **held bottom** (never
integrated), a thin **thermostatted buffer** just above it, and free `nve`
atoms everywhere else — so heat generated at the contact flows *out through
the base* instead of being damped away unphysically at the surface where the
action is.

In [ ]:
%pip install lammps-js matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps, LMP_STYLE_ATOM, LMP_TYPE_VECTOR

A0, TEMP, RADIUS, RATE = 1.5496, 0.05, 3.0, 0.08

lmp = await lammps(output=None)
lmp.commands_string(f"""
units lj
atom_style atomic
boundary p p f
lattice fcc {4 / A0**3:.8f}
region box block 0 8 0 8 -7 5 units lattice
create_box 1 box
region slab block INF INF INF INF -7 0 units lattice
create_atoms 1 region slab
mass 1 1.0
pair_style lj/cut 2.5
pair_coeff 1 1 1.0 1.0 2.5

variable zbot equal bound(all,zmin)
region rbot block INF INF INF INF INF $(v_zbot+1.2) units box
group bottom region rbot
group mobile subtract all bottom
region rbuf block INF INF INF INF $(v_zbot+1.2) $(v_zbot+2.8) units box
group buffer region rbuf

fix hold bottom setforce 0.0 0.0 0.0
fix intg mobile nve
fix thermostat buffer langevin {TEMP} {TEMP} 0.5 90210
timestep 0.005
velocity mobile create {TEMP} 4928459 mom yes
run 1000 post no
""")
print(lmp.get_natoms(), "atoms equilibrated at T =", round(lmp.get_thermo("temp"), 3))

## Drive the indenter down

`fix indent` adds a rigid repulsive sphere whose height is an equal-style
**variable** — LAMMPS re-evaluates it every step, so the sphere descends at a
constant velocity. Its reaction force `f_ind[3]` is the load:

In [ ]:
lmp.commands_string(f"""
variable ztop0 equal $(bound(all,zmax))
variable zind equal v_ztop0+{RADIUS}+0.3-step*dt*{RATE}
fix ind all indent 200.0 sphere $(lx/2) $(ly/2) v_zind {RADIUS} units box side out
variable load equal f_ind[3]
variable depth equal step*dt*{RATE}-0.3
run 0 post no
""")

depth, load = [], []
for chunk in range(22):
    lmp.command("run 300 post no")
    depth.append(lmp.extract_variable("depth"))
    load.append(lmp.extract_variable("load"))
depth, load = np.array(depth), np.array(load)
print(f"final depth {depth[-1]:.2f} sigma, final load {load[-1]:.1f} eps/sigma")

In [ ]:
plt.figure(figsize=(5.5, 3.6))
plt.plot(depth, load, "o-")
plt.xlabel("indenter depth below surface (σ)"); plt.ylabel("load (ε/σ)")
plt.title("Load–displacement curve")
plt.tight_layout(); plt.show()

The load rises steeply on contact (elastic, roughly Hertzian), and any
sudden **load drop is a "pop-in"** — a burst of dislocations nucleating under
the tip, letting the surface give way. (Run longer or faster to provoke
more of them.)

## The plastic zone under the tip

Color a side view by centrosymmetry: defects concentrate beneath the
contact:

In [ ]:
lmp.command("compute csym all centro/atom fcc")
lmp.command("run 0 post no")
csym = lmp.extract_compute("csym", LMP_STYLE_ATOM, LMP_TYPE_VECTOR)
x = lmp.extract_atom("x")

mid = np.abs(x[:, 1] - x[:, 1].mean()) < 3.0     # a slice through the center
plt.figure(figsize=(6, 4.5))
sc = plt.scatter(x[mid, 0], x[mid, 2], c=np.minimum(csym[mid], 10), s=22, cmap="coolwarm")
plt.colorbar(sc, label="centrosymmetry (capped at 10)")
plt.xlabel("x (σ)"); plt.ylabel("z (σ)")
plt.title("Slice under the indenter")
plt.gca().set_aspect("equal")
plt.tight_layout(); plt.show()

lmp.close()

**Exercises**
- Swap the sphere for a **flat punch**
  (`fix ind all indent 200.0 plane z v_zind hi units box`): the load rises
  much faster because the whole surface is compressed at once — compare with
  Figure 10 of the paper.
- Double `RADIUS`. Hertzian contact predicts the elastic slope scales like
  √R.
- Retract the indenter (make `RATE` negative after loading) and look for
  hysteresis: plasticity is irreversible.

---

That's the full arc of the series: a perfect crystal → its equation of
state → yield in a strained crystal → grain boundaries → fracture →
contact plasticity. Every input here transfers to production LAMMPS with a
real potential — swap `pair_style lj/cut` for `pair_style eam/alloy` and a
potential file, switch to `units metal`, and you are running the original
LiveCoMS tutorials.